# PyMMM Pipeline: ND2 → Zarr

This notebook demonstrates the full PyMMM pipeline:
1. Load an ND2 file lazily
2. Register (drift-correct) the data
3. Detect feeding lanes
4. Detect trenches within lanes
5. Extract trenches to a compressed zarr store

Each stage checkpoints to a companion zarr store, so you can restart the kernel and resume from any checkpoint.

In [ ]:
from pymmm import ND2Experiment, Registrator, LaneDetector, TrenchDetector, Extractor
from pymmm.checkpoint import CompanionStore
from dask.distributed import Client, LocalCluster
import hvplot.xarray
import hvplot

hvplot.extension('bokeh')

# Start a dask cluster for parallelisation
cluster = LocalCluster(processes=True, threads_per_worker=1, n_workers=8)
client = Client(cluster)
client

## 1. Load ND2 file

In [ ]:
exp = ND2Experiment("/path/to/data.nd2")  # ← Change this path
print(exp)

In [ ]:
# Interactive browse raw data
exp.data.hvplot.image(
    x="X", y="Y", cmap="Greys_r", dynamic=True,
    rasterize=True, widget_location="top", aspect="equal"
)

In [ ]:
# Optional: discard unwanted FOVs
# exp.discard_fovs(["xy029", "xy030"])

## 2. Companion store

In [ ]:
store = CompanionStore.for_experiment(exp)
store

## 3. Registration (Checkpoint 1)

Register the experiment to correct for stage drift.

In [ ]:
if store.has_registration():
    print("Loading registration from checkpoint...")
    reg = Registrator.load(exp, store)
else:
    reg = Registrator(
        exp, store,
        registration_channel="PC",  # ← Change to your phase-contrast channel
        mode="mean",
        rotation=0.9,               # ← Adjust rotation if needed
        roi={"y": (300, 900), "x": (0, -1)},
        mean_n_frames=10,
        mean_from="end",
    )
    reg.compute_mean_images(plot=True)
    reg.compute_tmats(plot=True)
    reg.save()

In [ ]:
# Interactive browse registered data
reg.get_stabilized_data().hvplot.image(
    x="X", y="Y", cmap="Greys_r", dynamic=True,
    rasterize=True, widget_location="top", aspect="equal"
)

In [ ]:
# Drift diagnostics for a specific FOV
reg.plot_drift(fov=0)

## 4. Lane Detection (Checkpoint 2)

Detect the feeding lane y-positions in each FOV.

In [ ]:
if store.has_lane_detection():
    print("Loading lane detection from checkpoint...")
    lane_det = LaneDetector.load(exp, reg, store)
else:
    lane_det = LaneDetector(exp, reg, store, detection_channel="PC")
    lane_det.detect_lanes(sigma=40, distance=300, height=5000, plot=True)
    lane_det.save()

In [ ]:
# Inspect another FOV
# lane_det.plot_fov(fov=1)

## 5. Trench Detection (Checkpoint 3)

Detect trench x-positions within each lane.

In [ ]:
if store.has_trench_detection():
    print("Loading trench detection from checkpoint...")
    trench_det = TrenchDetector.load(exp, reg, lane_det, store)
else:
    trench_det = TrenchDetector(exp, reg, lane_det, store, detection_channel="PC")
    trench_det.detect_trenches(
        sigma=4, distance=100, prominence=10,
        trench_width=128, trench_length=160,
        trench_bottom_offset=50, plot=True
    )
    # Optionally discard bad trenches:
    # trench_det.discard_trenches([20, 21, 30])
    trench_det.save()

In [ ]:
# Trench table
trench_det.get_trench_table()

In [ ]:
# Preview a single trench before full extraction
from pymmm.plotting import plot_trench_preview
preview = trench_det_extractor_preview = Extractor(exp, reg, trench_det)
single = preview.extract_single_trench(trench_id=0)
plot_trench_preview(single, trench_id=0, n_frames=8)

## 6. Extraction

Extract all trenches to a compressed zarr store.

In [ ]:
extractor = Extractor(exp, reg, trench_det)
extractor.extract(compressor='zstd', clevel=9, show_progress=True)

## 7. Verify output

In [ ]:
import zarr

z = zarr.open(str(extractor.output_path), mode='r')
print("Shape:", z['data'].shape)
print("Chunks:", z['data'].chunks)
print("Attrs:", dict(z.attrs))